# Nodes & Documents — LlamaIndex's Core Data Model Explained

Two terms come up constantly in LlamaIndex and are easy to conflate:

- A **Document** represents one whole source — a file, a webpage, a database row.
- A **Node** represents one chunk of that document after it's been split, which is the actual unit LlamaIndex embeds, stores, and retrieves.

Every index you've built so far did this splitting automatically behind the scenes. This episode does it manually so you can see exactly what a Document turns into.


**Step 1 — Create a Document by hand.** Instead of loading from `data/sample_docs`, this builds a `Document` manually from a raw string of Naruto lore with custom metadata attached, so you can see a Document's structure before it ever gets split into Nodes.


In [1]:
from llama_index.core import Document

raw_text = """Naruto — Tailed Beasts and Jinchuriki Explained

In the Naruto universe, Tailed Beasts (Bijuu) are nine enormous, ancient chakra beings, each possessing a number of tails from one to nine that roughly corresponds to the sheer scale of their power. They were originally a single entity, the Ten-Tails, before the sage Hagoromo Otsutsuki split its chakra into the nine separate beasts to prevent its destructive power from ever being wielded as one again. This document explains how Tailed Beasts relate to their human hosts, known as jinchuriki, and why that relationship is central to the series' plot.

What is a jinchuriki
A jinchuriki, literally meaning "power of human sacrifice," is a person who has a Tailed Beast sealed inside them, usually through a technique performed by a skilled sealing specialist. The sealing process is dangerous and historically often fatal for the host, though later generations developed safer sealing methods. Jinchuriki generally gain access to some fraction of their beast's chakra, granting them abilities far beyond a typical ninja, but at the cost of carrying an entity that, especially early in the host relationship, often resents being imprisoned.

Naruto and Kurama
Naruto Uzumaki is the jinchuriki of Kurama, the Nine-Tailed Fox, the most powerful of the nine Tailed Beasts. Kurama was sealed inside Naruto as an infant by his father, the Fourth Hokage, during an attack on the Hidden Leaf Village — the same attack that killed both of Naruto's parents. For much of Naruto's early life, Kurama is hostile toward him, having been manipulated for years by other characters into attacking the village. Their relationship gradually shifts from imprisoner-and-prisoner into a genuine partnership, culminating in Kurama willingly lending Naruto its full chakra during the Fourth Great Ninja War.

Why villages hunt jinchuriki
The organization Akatsuki spends much of the series capturing jinchuriki one by one in order to extract their Tailed Beasts, since combining all nine beasts' chakra would let its members recreate the Ten-Tails and cast a reality-altering genjutsu over the entire world. This hunt is the throughline connecting most of the series' major arcs, since nearly every major village loses a jinchuriki to Akatsuki over the course of the story, raising the stakes of Naruto's own situation as one of the last remaining hosts.

Other notable jinchuriki
Gaara, jinchuriki of the One-Tailed Shukaku, begins the series as an antagonist shaped by a childhood nearly identical to Naruto's own isolation, but follows a redemption arc that turns him into one of Naruto's closest allies and eventually the Fifth Kazekage of the Hidden Sand Village. Killer Bee, jinchuriki of the Eight-Tailed Gyuki, is notable for having achieved a fully cooperative relationship with his Tailed Beast well before the series begins, serving as a template Naruto later follows with Kurama.

Why this lore matters
Understanding the Tailed Beasts explains much of what looks, at first glance, like arbitrary power scaling in later arcs: a ninja's raw strength in the back half of the series is frequently tied directly to whether they have Tailed Beast chakra to draw on, and the eventual thawing of hostility between jinchuriki and their beasts across multiple characters mirrors the series' broader theme of breaking inherited cycles of fear and hatred."""

# Document is LlamaIndex's container for one whole source, plus any metadata you
# attach — here a fake "source" and "category" standing in for real corpus metadata.
document = Document(
    text=raw_text,
    metadata={"source": "manual_entry", "category": "anime_lore"},
)
print(f"Document ID: {document.doc_id}")  # auto-generated unique id for this Document
print(f"Metadata: {document.metadata}")
print(f"Text length: {len(document.text)} characters")

Document ID: c61ee3eb-b086-45e7-8e9d-408b48f38f4f
Metadata: {'source': 'manual_entry', 'category': 'anime_lore'}
Text length: 3382 characters


**Step 2 — Split it into Nodes.** `SentenceSplitter` is the component that does the chunking `from_documents()` normally performs for you automatically behind the scenes. Notice each resulting `Node` inherits the parent Document's metadata for free.


In [2]:
from llama_index.core.node_parser import SentenceSplitter

# SentenceSplitter breaks text into chunks close to chunk_size characters, snapping
# to sentence boundaries where possible, with chunk_overlap characters shared
# between consecutive chunks so context isn't cut off mid-thought.
splitter = SentenceSplitter(chunk_size=200, chunk_overlap=20)
nodes = splitter.get_nodes_from_documents([document])

print(f"Split into {len(nodes)} nodes\n")
for i, node in enumerate(nodes):
    # Nodes automatically inherit their parent Document's metadata — no manual copying needed.
    print(f"--- Node {i} | metadata inherited from Document: {node.metadata} ---")
    print(node.get_content()[:150] + "...\n")

Split into 5 nodes

--- Node 0 | metadata inherited from Document: {'source': 'manual_entry', 'category': 'anime_lore'} ---
Naruto — Tailed Beasts and Jinchuriki Explained

In the Naruto universe, Tailed Beasts (Bijuu) are nine enormous, ancient chakra beings, each possessi...

--- Node 1 | metadata inherited from Document: {'source': 'manual_entry', 'category': 'anime_lore'} ---
The sealing process is dangerous and historically often fatal for the host, though later generations developed safer sealing methods. Jinchuriki gener...

--- Node 2 | metadata inherited from Document: {'source': 'manual_entry', 'category': 'anime_lore'} ---
Their relationship gradually shifts from imprisoner-and-prisoner into a genuine partnership, culminating in Kurama willingly lending Naruto its full c...

--- Node 3 | metadata inherited from Document: {'source': 'manual_entry', 'category': 'anime_lore'} ---
Other notable jinchuriki
Gaara, jinchuriki of the One-Tailed Shukaku, begins the series as an antag

**Step 3 — See the chunk size/overlap trade-off.** Re-splitting the exact same document at three different `chunk_size` values shows directly how that one setting controls both how many nodes you get and how much text each one holds.


In [3]:
from llama_index.core.node_parser import SentenceSplitter

# Re-split the same document with different chunk_size/chunk_overlap combos to see
# how those two settings trade off node count against how much text each node holds.
for chunk_size, chunk_overlap in [(200, 20), (500, 50), (1024, 20)]:
    splitter = SentenceSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    resized_nodes = splitter.get_nodes_from_documents([document])
    avg_len = sum(len(n.get_content()) for n in resized_nodes) / len(resized_nodes)
    print(
        f"chunk_size={chunk_size}, chunk_overlap={chunk_overlap} -> "
        f"{len(resized_nodes)} nodes, avg {avg_len:.0f} chars each"
    )

chunk_size=200, chunk_overlap=20 -> 5 nodes, avg 675 chars each
chunk_size=500, chunk_overlap=50 -> 2 nodes, avg 1690 chars each
chunk_size=1024, chunk_overlap=20 -> 1 nodes, avg 3382 chars each


### Summary

- **Documents** are whole sources; **Nodes** are the chunks LlamaIndex actually indexes and retrieves, and they inherit their parent Document's metadata automatically.
- `chunk_size` and `chunk_overlap` directly control how many nodes you get and how much context each one holds — too small and you lose context, too large and retrieval gets less precise. We'll see this trade-off matter more once we start tuning retrieval quality in later episodes.
